In [ ]:
# Decision Tree Classification for Machine Failure Prediction

This notebook builds a Decision Tree classifier to predict whether a machine will fail using the provided maintenance dataset.

SyntaxError: invalid syntax (2088446755.py, line 3)

In [ ]:
import warnings
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

In [ ]:
# Load and inspect the dataset
DATA_PATH = "machine_failure_data.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nFirst 10 rows:")
print(df.head(10).to_string(index=False))
print("\nLast 10 rows:")
print(df.tail(10).to_string(index=False))
print("\nColumn data types:")
print(df.dtypes)
print("\nDataset loaded successfully.")

## Data Cleaning and Preprocessing

The identifier column is removed, duplicate rows are dropped, and missing values are checked.

In [ ]:
initial_shape = df.shape

df = df.drop(columns=["UDI"])

before_duplicates = df.shape[0]
df = df.drop_duplicates()
after_duplicates = df.shape[0]
removed_duplicates = before_duplicates - after_duplicates

missing_values = df.isnull().sum()

print("Shape after dropping UDI:", df.shape)
print("Duplicate rows removed:", removed_duplicates)
print("\nMissing values per column:")
print(missing_values)

In [ ]:
# Exploratory data analysis
print("\nNumerical summary:")
print(df.describe().T)

categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
print("\nCategorical summary:")
if categorical_cols:
    print(df[categorical_cols].describe(include="all"))
else:
    print("No categorical columns found.")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df, x="Torque", hue="Failure", kde=True, ax=axes[0], alpha=0.6)
axes[0].set_title("Torque Distribution by Failure Status")
sns.countplot(data=df, x="Type", hue="Failure", ax=axes[1])
axes[1].set_title("Failure Count by Machine Type")
plt.tight_layout()
plt.show()

In [ ]:
# Train-test split
X = df.drop(columns=["Failure"])
y = df["Failure"]

X = pd.get_dummies(X, columns=["Type"], drop_first=False)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)
print("Target distribution in training set:\n", y_train.value_counts(normalize=True))

In [ ]:
# Build and train the decision tree
clf = DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=4)
clf.fit(X_train, y_train)

print("Decision Tree trained successfully.")

In [ ]:
# Evaluate the model
y_pred = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba)
cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))
print("ROC-AUC:", round(roc_auc, 4))
print("\nConfusion Matrix:")
print(cm)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Tree visualization
plt.figure(figsize=(18, 8))
plot_tree(
    clf,
    feature_names=list(X.columns),
    class_names=["No Failure", "Failure"],
    filled=True,
    rounded=True,
    max_depth=3,
)
plt.title("Decision Tree for Machine Failure Prediction")
plt.show()

print("\nTree structure (text view):")
print(export_text(clf, feature_names=list(X.columns)))

print("\nObservations:")
print("- The dataset contains 10,000 rows and 8 columns before preprocessing.")
print("- The UDI identifier column was removed because it does not contribute predictive information.")
print("- No duplicate rows were found after cleaning.")
print("- Missing values were not present in the cleaned dataset.")
print("- The Decision Tree model provides a compact, interpretable structure for failure prediction.")